In [2]:
# =============================================================================
# ASCENT Phase 1 — train.py
# Train a small MLP on MNIST, prune to 60% sparsity, fine-tune, save weights.
#
# Kaggle usage:
#   1. Upload this file to /kaggle/working/
#   2. In a notebook cell run:  %run /kaggle/working/train.py
#   3. Output goes to /kaggle/working/outputs/model.pth
# =============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import os

# -----------------------------------------------------------------------------
# Output directory — works on both Kaggle and local machines
# -----------------------------------------------------------------------------
OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# GPU detection — Kaggle's T4 makes training ~10x faster than CPU.
# -----------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------------
# SECTION 1: Hyperparameters
# A hyperparameter is a setting you choose before training — not learned.
# -----------------------------------------------------------------------------

BATCH_SIZE   = 64     # Images per batch before weight update.
EPOCHS_TRAIN = 10     # Full passes over training set.
EPOCHS_FINE  = 5      # Fine-tuning epochs after pruning.
LR           = 1e-3   # Learning rate (Adam default).
SPARSITY     = 0.60   # Zero out bottom 60% of weights by magnitude.
SEED         = 42     # Reproducibility.

torch.manual_seed(SEED)

# -----------------------------------------------------------------------------
# SECTION 2: Data Loading
# transforms.Compose chains transforms.
# ToTensor()    : PIL image (0-255 uint8) → float32 tensor (0.0-1.0)
# Normalize()   : (x - mean) / std. Numbers are precomputed MNIST stats.
# -----------------------------------------------------------------------------

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples : {len(train_dataset)}")
print(f"Test samples     : {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

# -----------------------------------------------------------------------------
# SECTION 3: Model Definition
# nn.Module is PyTorch's base class for networks.
# Subclass it: define layers in __init__, define data flow in forward().
# -----------------------------------------------------------------------------

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear(in, out): fully-connected layer.
        # Stores weight matrix (out × in) + bias vector (out).
        # Computes  y = x @ W.T + b   — exactly one layer of MAC operations.
        self.fc1  = nn.Linear(784, 128)
        self.fc2  = nn.Linear(128, 64)
        self.fc3  = nn.Linear(64,  10)
        self.relu = nn.ReLU()    # max(0, x), no parameters

    def forward(self, x):
        x = x.view(-1, 784)              # flatten (batch, 1, 28, 28) → (batch, 784)
        x = self.relu(self.fc1(x))       # Layer 1 + ReLU
        x = self.relu(self.fc2(x))       # Layer 2 + ReLU
        x = self.fc3(x)                  # Layer 3, no activation (CrossEntropyLoss
                                         # applies softmax internally)
        return x                         # (batch, 10) logits

model = MLP().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nMLP total parameters: {total_params:,}")

# -----------------------------------------------------------------------------
# SECTION 4: Loss + Optimizer
# -----------------------------------------------------------------------------

criterion = nn.CrossEntropyLoss()                   # softmax + neg-log-likelihood
optimizer = optim.Adam(model.parameters(), lr=LR)   # adaptive gradient optimizer

# -----------------------------------------------------------------------------
# SECTION 5: Train + Evaluate functions
# -----------------------------------------------------------------------------

def train_one_epoch(epoch_num):
    model.train()    # training mode
    total_loss = 0.0
    correct    = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()              # clear previous gradients
        output = model(images)             # forward pass
        loss   = criterion(output, labels) # scalar loss
        loss.backward()                    # backprop
        optimizer.step()                   # update weights

        total_loss += loss.item()
        _, predicted = torch.max(output, dim=1)
        correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = 100.0 * correct / len(train_dataset)
    print(f"  Epoch {epoch_num:2d} | Loss: {avg_loss:.4f} | Train Acc: {accuracy:.2f}%")
    return accuracy


def evaluate():
    model.eval()
    correct = 0
    with torch.no_grad():    # no gradient tracking — faster, less memory
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model(images)
            _, predicted = torch.max(output, dim=1)
            correct += (predicted == labels).sum().item()
    accuracy = 100.0 * correct / len(test_dataset)
    print(f"           Test  Acc: {accuracy:.2f}%")
    return accuracy

# -----------------------------------------------------------------------------
# SECTION 6: Initial Training
# -----------------------------------------------------------------------------

print("\n--- Phase 1a: Initial Training ---")
for epoch in range(1, EPOCHS_TRAIN + 1):
    train_one_epoch(epoch)
evaluate()

# -----------------------------------------------------------------------------
# SECTION 7: Pruning
# Set the smallest-magnitude 60% of weights to zero. Use a global threshold
# across all layers (better than per-layer for small networks).
# Prune only weight matrices, not biases.
# -----------------------------------------------------------------------------

print("\n--- Phase 1b: Pruning ---")

def apply_pruning(model, sparsity):
    all_weights = []
    for name, param in model.named_parameters():
        if 'weight' in name:
            all_weights.append(param.data.abs().flatten())
    all_weights = torch.cat(all_weights)

    # Value below which `sparsity` fraction of weights fall.
    threshold = torch.quantile(all_weights, sparsity)
    print(f"  Pruning threshold (magnitude): {threshold:.6f}")

    masks = {}
    total = zeroed = 0
    for name, param in model.named_parameters():
        if 'weight' in name:
            mask = (param.data.abs() >= threshold).float()
            param.data *= mask
            masks[name] = mask
            total  += mask.numel()
            zeroed += (mask == 0).sum().item()

    print(f"  Actual sparsity: {100*zeroed/total:.1f}%  ({zeroed}/{total} weights zeroed)")
    return masks

masks = apply_pruning(model, SPARSITY)
print("  Accuracy immediately after pruning:")
evaluate()

# -----------------------------------------------------------------------------
# SECTION 8: Fine-tuning
# Train for EPOCHS_FINE more epochs while keeping pruned weights at zero.
# Zero gradients of pruned weights so optimizer never restores them.
# -----------------------------------------------------------------------------

print("\n--- Phase 1c: Fine-tuning ---")

for epoch in range(1, EPOCHS_FINE + 1):
    model.train()
    total_loss = 0.0
    correct = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()

        # Mask gradients before optimizer step
        with torch.no_grad():
            for name, param in model.named_parameters():
                if 'weight' in name and name in masks:
                    param.grad *= masks[name]

        optimizer.step()

        # Re-enforce zeros (float ops can introduce tiny residuals)
        with torch.no_grad():
            for name, param in model.named_parameters():
                if 'weight' in name and name in masks:
                    param.data *= masks[name]

        total_loss += loss.item()
        _, predicted = torch.max(output, dim=1)
        correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = 100.0 * correct / len(train_dataset)
    print(f"  Fine-tune Epoch {epoch} | Loss: {avg_loss:.4f} | Train Acc: {accuracy:.2f}%")

print("  Final accuracy after fine-tuning:")
final_acc = evaluate()

# Verify final sparsity
total = zeroed = 0
for name, param in model.named_parameters():
    if 'weight' in name:
        total  += param.numel()
        zeroed += (param.data == 0).sum().item()
print(f"\nFinal sparsity: {100*zeroed/total:.1f}%  ({zeroed}/{total} weights are zero)")

# -----------------------------------------------------------------------------
# SECTION 9: Save
# state_dict() = dictionary of all parameter tensors. Standard PyTorch save.
# Move masks to CPU so they load anywhere later.
# -----------------------------------------------------------------------------

masks_cpu = {k: v.cpu() for k, v in masks.items()}

save_path = os.path.join(OUTPUT_DIR, 'model.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'masks':            masks_cpu,
    'final_accuracy':   final_acc,
    'sparsity':         SPARSITY,
}, save_path)

print(f"\nSaved → {save_path}")
print(f"Final test accuracy: {final_acc:.2f}%")
print("Next: run quantize.py")

Device: cuda
GPU: Tesla T4
Training samples : 60000
Test samples     : 10000
Batches per epoch: 938

MLP total parameters: 109,386

--- Phase 1a: Initial Training ---
  Epoch  1 | Loss: 0.2645 | Train Acc: 92.16%
  Epoch  2 | Loss: 0.1109 | Train Acc: 96.63%
  Epoch  3 | Loss: 0.0771 | Train Acc: 97.59%
  Epoch  4 | Loss: 0.0614 | Train Acc: 98.03%
  Epoch  5 | Loss: 0.0483 | Train Acc: 98.44%
  Epoch  6 | Loss: 0.0388 | Train Acc: 98.76%
  Epoch  7 | Loss: 0.0336 | Train Acc: 98.87%
  Epoch  8 | Loss: 0.0294 | Train Acc: 99.00%
  Epoch  9 | Loss: 0.0262 | Train Acc: 99.08%
  Epoch 10 | Loss: 0.0249 | Train Acc: 99.14%
           Test  Acc: 97.78%

--- Phase 1b: Pruning ---
  Pruning threshold (magnitude): 0.053800
  Actual sparsity: 60.0%  (65510/109184 weights zeroed)
  Accuracy immediately after pruning:
           Test  Acc: 97.50%

--- Phase 1c: Fine-tuning ---
  Fine-tune Epoch 1 | Loss: 0.0117 | Train Acc: 99.62%
  Fine-tune Epoch 2 | Loss: 0.0058 | Train Acc: 99.82%
  Fine-tune

In [3]:
# =============================================================================
# ASCENT Phase 1 — quantize.py
# Load trained model, apply INT8 symmetric quantization, export:
#   outputs/weights_l1.hex  — Layer 1 weights
#   outputs/weights_l2.hex  — Layer 2 weights
#   outputs/weights_l3.hex  — Layer 3 weights
#   outputs/scales.json     — Per-layer scale factors
#
# .hex format: one 8-bit value per line, in two's complement hex.
# Verilog $readmemh loads these directly into the CIM array memory.
# =============================================================================

import torch
import torch.nn as nn
import numpy as np
import json
import os

# Path detection — Kaggle vs local
OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'

# Redefine the model class — must match train.py exactly so state_dict loads.
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(784, 128)
        self.fc2  = nn.Linear(128, 64)
        self.fc3  = nn.Linear(64,  10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# -----------------------------------------------------------------------------
# SECTION 1: Load trained model
# -----------------------------------------------------------------------------

ckpt_path = os.path.join(OUTPUT_DIR, 'model.pth')
checkpoint = torch.load(ckpt_path, map_location='cpu')

model = MLP()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded model. Final training accuracy: {checkpoint['final_accuracy']:.2f}%")

# -----------------------------------------------------------------------------
# SECTION 2: Quantization function
#
# Symmetric per-layer INT8 quantization:
#   scale  = max(|W|) / 127
#   W_int8 = clip( round(W / scale), -128, 127 )
#
# "Symmetric" → zero in float maps to zero in INT8 (no zero-point offset).
# Critical for hardware: keeps trained zeros at exactly zero, preserving sparsity.
# -----------------------------------------------------------------------------

def quantize_layer(weight_tensor):
    """
    weight_tensor: float32 tensor, shape (out_features, in_features)
    Returns:
        w_int8 : int8 numpy array, same shape
        scale  : float scalar
    """
    W = weight_tensor.detach().numpy().astype(np.float32)
    max_abs = np.max(np.abs(W))

    if max_abs == 0:
        return np.zeros_like(W, dtype=np.int8), 1.0

    scale = max_abs / 127.0
    W_int8 = np.clip(np.round(W / scale), -128, 127).astype(np.int8)
    return W_int8, scale


# -----------------------------------------------------------------------------
# SECTION 3: Quantize all three layers
# -----------------------------------------------------------------------------

layers = [
    ('fc1', model.fc1.weight, os.path.join(OUTPUT_DIR, 'weights_l1.hex')),
    ('fc2', model.fc2.weight, os.path.join(OUTPUT_DIR, 'weights_l2.hex')),
    ('fc3', model.fc3.weight, os.path.join(OUTPUT_DIR, 'weights_l3.hex')),
]

scales = {}
os.makedirs(OUTPUT_DIR, exist_ok=True)

for layer_name, weight_tensor, hex_path in layers:
    W_int8, scale = quantize_layer(weight_tensor)
    scales[layer_name] = float(scale)

    # Sparsity report
    total  = W_int8.size
    zeroed = int(np.sum(W_int8 == 0))
    print(f"\n{layer_name}: shape={W_int8.shape}  scale={scale:.6f}")
    print(f"  Sparsity after quantization: {100*zeroed/total:.1f}%  ({zeroed}/{total})")

    # Write .hex file.
    # Each weight = one INT8 value written as 2 hex digits (two's complement).
    #   -1   → 0xFF        -128 → 0x80
    #    0   → 0x00         127 → 0x7F
    # np.int8 → view as uint8 gives the raw bit pattern, exactly what
    # $readmemh expects for signed values.
    #
    # Layout: row-major (output neuron first).
    # For Layer 1 (128×784): row i starts at address i*784.
    flat = W_int8.flatten().view(np.uint8)

    with open(hex_path, 'w') as f:
        for val in flat:
            f.write(f"{val:02x}\n")

    print(f"  Written → {hex_path}  ({len(flat)} entries)")

# -----------------------------------------------------------------------------
# SECTION 4: Save scales
# Hardware doesn't use these — it works in pure INT8.
# Golden model uses them to convert outputs back to float for accuracy check.
# -----------------------------------------------------------------------------

scales_path = os.path.join(OUTPUT_DIR, 'scales.json')
with open(scales_path, 'w') as f:
    json.dump(scales, f, indent=2)
print(f"\nScales saved → {scales_path}")
print(json.dumps(scales, indent=2))

# -----------------------------------------------------------------------------
# SECTION 5: Quantization accuracy check
# Run quantized weights (as float) through MNIST test set.
# Should be within ~1% of original float32 accuracy.
# -----------------------------------------------------------------------------

print("\n--- Quantization Accuracy Check ---")

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Reconstruct quantized weights as floats for fast batch evaluation
q_layers = {}
for layer_name, weight_tensor, hex_path in layers:
    W_int8, scale = quantize_layer(weight_tensor)
    q_layers[layer_name] = (W_int8.astype(np.float32) * scales[layer_name])

def quantized_forward(x_np):
    x = x_np.reshape(-1, 784)
    x = x @ q_layers['fc1'].T
    x = np.maximum(0, x)
    x = x @ q_layers['fc2'].T
    x = np.maximum(0, x)
    x = x @ q_layers['fc3'].T
    return x

correct = total = 0
for images, labels in test_loader:
    x_np   = images.numpy().reshape(-1, 784)
    logits = quantized_forward(x_np)
    pred   = np.argmax(logits, axis=1)
    correct += int(np.sum(pred == labels.numpy()))
    total   += len(labels)

quant_acc = 100.0 * correct / total
float_acc = checkpoint['final_accuracy']
drop      = float_acc - quant_acc

print(f"Float32 accuracy  : {float_acc:.2f}%")
print(f"Quantized accuracy: {quant_acc:.2f}%")
print(f"Accuracy drop     : {drop:.2f}%")

if drop < 2.0:
    print("PASS: Quantization drop within acceptable range (<2%)")
else:
    print("WARN: Quantization drop >2%. Consider QAT (quantization-aware training).")

print("\nNext: run golden_model.py")

Loaded model. Final training accuracy: 97.85%

fc1: shape=(128, 784)  scale=0.006771
  Sparsity after quantization: 62.3%  (62546/100352)
  Written → /kaggle/working/outputs/weights_l1.hex  (100352 entries)

fc2: shape=(64, 128)  scale=0.004826
  Sparsity after quantization: 36.7%  (3009/8192)
  Written → /kaggle/working/outputs/weights_l2.hex  (8192 entries)

fc3: shape=(10, 64)  scale=0.005061
  Sparsity after quantization: 23.3%  (149/640)
  Written → /kaggle/working/outputs/weights_l3.hex  (640 entries)

Scales saved → /kaggle/working/outputs/scales.json
{
  "fc1": 0.006771059241145849,
  "fc2": 0.004825628828257322,
  "fc3": 0.005061395466327667
}

--- Quantization Accuracy Check ---
Float32 accuracy  : 97.85%
Quantized accuracy: 97.85%
Accuracy drop     : 0.00%
PASS: Quantization drop within acceptable range (<2%)

Next: run golden_model.py


In [4]:
# =============================================================================
# ASCENT Phase 1 — golden_model.py
# Simulate the hardware's exact integer arithmetic in Python.
# This is the ground truth every RTL testbench checks against.
#
# Key principle: this file must match the hardware EXACTLY.
# Same bit widths, same overflow behaviour, same rounding.
#
# Outputs:
#   outputs/golden_outputs.txt  — per-image: index, true label, prediction, scores
#   outputs/test_inputs.hex     — raw INT8 pixel values for 100 test images
# =============================================================================

import numpy as np
import json
import os
from torchvision import datasets

# Path detection
OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'

# -----------------------------------------------------------------------------
# SECTION 1: Load quantized weights from .hex files
# -----------------------------------------------------------------------------

def load_hex_weights(path, shape):
    """
    Load a weight hex file produced by quantize.py.
    Each line is one byte in two's complement hex.
    Returns int8 numpy array reshaped to `shape`.
    """
    values = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('//'):
                # int(x, 16) → uint8, then .view(int8) reinterprets the bits as signed
                val = np.array([int(line, 16)], dtype=np.uint8).view(np.int8)[0]
                values.append(val)
    return np.array(values, dtype=np.int8).reshape(shape)


# Layer shapes: (output_neurons, input_neurons)
W1 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l1.hex'), (128, 784))
W2 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l2.hex'), ( 64, 128))
W3 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l3.hex'), ( 10,  64))

print(f"Loaded weights: L1{W1.shape}  L2{W2.shape}  L3{W3.shape}")
print(f"L1 sparsity: {100*np.mean(W1==0):.1f}%")
print(f"L2 sparsity: {100*np.mean(W2==0):.1f}%")
print(f"L3 sparsity: {100*np.mean(W3==0):.1f}%")

# Load scales
with open(os.path.join(OUTPUT_DIR, 'scales.json')) as f:
    scales = json.load(f)

# -----------------------------------------------------------------------------
# SECTION 2: Integer MAC simulation
#
# Hardware computes:  acc = sum_i( x[i] * w[i] ),  x and w are INT8 signed.
# INT8 × INT8 product fits in 16 bits.
# Summing 128 such products needs ~7 more bits → 23 bits worst case.
# Hardware budgets 20 bits with mild saturation (acceptable for NNs).
#
# We use int32 in Python to mirror hardware faithfully, then clip to 20b.
#
# ReLU re-quantization: after summing, the accumulator is much wider than
# INT8. After ReLU we re-scale back to INT8 for the next layer's input.
# -----------------------------------------------------------------------------

ACC_MIN = -524288     # -2^19
ACC_MAX =  524287     #  2^19 - 1

def integer_layer(x_int8, W_int8, scale_in, scale_w, clip_output=True):
    """
    One MLP layer in integer arithmetic.

    x_int8     : (n_in,)   INT8 input vector
    W_int8     : (n_out,n_in) INT8 weight matrix
    scale_in   : float scale of inputs
    scale_w    : float scale of weights
    clip_output: True  → apply ReLU + re-quantize to INT8 (returns int8, scale)
                 False → return raw 20-bit accumulator (returns int32, None)
    """
    n_out, n_in = W_int8.shape
    acc = np.zeros(n_out, dtype=np.int32)

    # Explicit row-by-row loop mirrors the CIM array structure.
    # Each row of W_int8 = one output neuron = 128 weights stored in CIM.
    for j in range(n_out):
        for i in range(n_in):
            # The exact MAC the hardware performs
            acc[j] += int(x_int8[i]) * int(W_int8[j, i])
        # Saturate to 20-bit signed
        acc[j] = int(np.clip(acc[j], ACC_MIN, ACC_MAX))

    if not clip_output:
        # Last layer: raw scores out, no activation
        return acc, None

    # ReLU
    acc_relu = np.maximum(0, acc)

    # Re-quantize to INT8 for next layer.
    # Using max-of-batch as the new scale keeps accuracy high.
    max_val = np.max(acc_relu)
    if max_val == 0:
        return np.zeros(n_out, dtype=np.int8), scale_in * scale_w

    requant_scale = max_val / 127.0
    out_int8 = np.clip(np.round(acc_relu / requant_scale), 0, 127).astype(np.int8)
    return out_int8, requant_scale


def run_inference(pixel_int8):
    """
    Run one image through the integer MLP.
    pixel_int8: (784,) INT8 array
    Returns:
        predicted (int): predicted digit class 0..9
        scores (int32 array length 10): raw layer-3 outputs
    """
    out1, scale1 = integer_layer(pixel_int8, W1,
                                 scale_in=1.0,
                                 scale_w=scales['fc1'],
                                 clip_output=True)

    out2, scale2 = integer_layer(out1, W2,
                                 scale_in=scale1,
                                 scale_w=scales['fc2'],
                                 clip_output=True)

    scores, _ = integer_layer(out2, W3,
                              scale_in=scale2,
                              scale_w=scales['fc3'],
                              clip_output=False)

    return int(np.argmax(scores)), scores


# -----------------------------------------------------------------------------
# SECTION 3: Run on 100 test images, write golden outputs
# -----------------------------------------------------------------------------

test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True)

NUM_IMAGES = 100
correct = 0

inputs_hex_path = os.path.join(OUTPUT_DIR, 'test_inputs.hex')
golden_path     = os.path.join(OUTPUT_DIR, 'golden_outputs.txt')

print(f"\nRunning integer inference on {NUM_IMAGES} test images...")
print("(This is slow — ~10 sec per image due to explicit MAC loops.)")

inputs_hex = open(inputs_hex_path, 'w')

with open(golden_path, 'w') as f_out:
    f_out.write("# ASCENT Golden Model Outputs\n")
    f_out.write("# Format: index  true_label  predicted  [score_0..score_9]  status\n")

    for idx in range(NUM_IMAGES):
        img_pil, true_label = test_dataset[idx]
        # Raw uint8 pixel values, 0..255
        img_np = np.array(img_pil, dtype=np.uint8).flatten()

        # Write pixel bytes to test_inputs.hex
        for px in img_np:
            inputs_hex.write(f"{px:02x}\n")

        # Run integer inference (cast uint8 to int8 — pixels 128..255 become negatives,
        # but the hardware sees them the same way, and the network was trained accordingly)
        predicted, scores = run_inference(img_np.astype(np.int8))

        if predicted == true_label:
            correct += 1

        scores_str = ' '.join(str(int(s)) for s in scores)
        status = 'OK' if predicted == true_label else 'WRONG'
        f_out.write(f"{idx:4d}  {true_label}  {predicted}  [{scores_str}]  {status}\n")

        if idx < 10:
            print(f"  [{idx:3d}] true={true_label}  pred={predicted}  {status}  scores={scores}")
        elif idx % 20 == 0:
            print(f"  [{idx:3d}] running...")

inputs_hex.close()

accuracy = 100.0 * correct / NUM_IMAGES
print(f"\nGolden model accuracy on {NUM_IMAGES} images: {accuracy:.1f}%")
print(f"Written → {golden_path}")
print(f"Written → {inputs_hex_path}")

if accuracy >= 90.0:
    print("\nPASS: Golden model accuracy acceptable (>=90%)")
    print("Phase 1 complete. Outputs ready for RTL verification.")
    print("Next: Phase 2 — LOA Approximate Multiplier")
else:
    print("\nWARN: Accuracy below 90%. Check quantize.py re-quantization logic.")

Loaded weights: L1(128, 784)  L2(64, 128)  L3(10, 64)
L1 sparsity: 62.3%
L2 sparsity: 36.7%
L3 sparsity: 23.3%

Running integer inference on 100 test images...
(This is slow — ~10 sec per image due to explicit MAC loops.)
  [  0] true=7  pred=7  OK  scores=[-27253  -5637  -5003 -13317 -26873   9646 -11294  19113 -20432 -15168]
  [  1] true=2  pred=7  WRONG  scores=[-12651   3226 -17035 -10579 -24406   -580 -19639  26854 -40541 -19708]
  [  2] true=1  pred=0  WRONG  scores=[ 33089 -21815   5696 -14194 -17587 -33964   4099  -9555 -18830 -12736]
  [  3] true=0  pred=7  WRONG  scores=[-40194   -104   4409   3222 -34186  18952 -28593  20307 -25815 -37901]
  [  4] true=4  pred=2  WRONG  scores=[ -4842   -376  18570  -7768 -31040  -6402   1519  -3807 -17222 -33254]
  [  5] true=1  pred=7  WRONG  scores=[-12295  -8494  -9929 -17248 -25410  12412 -14595  13906 -36623  -6798]
  [  6] true=4  pred=1  WRONG  scores=[-24649  21469  -3142 -26167  -7183   4555  -1054  12088 -34123 -44358]
  [  7] tru

In [5]:
# =============================================================================
# ASCENT Phase 1 — golden_model.py  (FIXED v2)
# Simulate the hardware's exact integer arithmetic in Python.
#
# BUGS FIXED FROM v1:
#
# BUG 1 — Per-image re-quantisation scale
#   v1 computed requant_scale = max_val / 127 for each image individually.
#   This meant every image got a different scale between layers 1→2 and 2→3.
#   Layer 2 received values numerically in 0..127 but semantically meaningless
#   — a quiet image and a loud image both collapsed to the same range,
#   destroying all relative magnitude information across the batch.
#   Result: 10% accuracy (pure random guessing).
#
#   FIX: Remove inter-layer re-quantisation entirely. Use int32 accumulators
#   all the way through all three layers. The golden model only needs argmax
#   of the final scores — absolute scale is irrelevant, only ordering matters.
#
# BUG 2 — Raw uint8 pixels cast directly to int8
#   v1 did: img_np.astype(np.int8)
#   This reinterprets bits, wrapping pixels 128..255 to -128..-1.
#   But the network was trained on NORMALISED float inputs:
#       normalised = (pixel/255.0 - 0.1307) / 0.3081
#   The INT8 inputs to the hardware must encode these normalised values.
#
#   FIX: Normalise pixels first with the same constants used in train.py,
#   then quantise the normalised float to INT8 with a fixed INPUT_SCALE.
#   test_inputs.hex now stores these normalised INT8 values.
# =============================================================================

import numpy as np
import json
import os
from torchvision import datasets

OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'

# -----------------------------------------------------------------------------
# SECTION 1: Load quantized weights from .hex files
# -----------------------------------------------------------------------------

def load_hex_weights(path, shape):
    values = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                val = np.array([int(line, 16)], dtype=np.uint8).view(np.int8)[0]
                values.append(val)
    return np.array(values, dtype=np.int8).reshape(shape)


W1 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l1.hex'), (128, 784))
W2 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l2.hex'), ( 64, 128))
W3 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l3.hex'), ( 10,  64))

print(f"Loaded weights:  L1{W1.shape}  L2{W2.shape}  L3{W3.shape}")
print(f"L1 sparsity: {100*np.mean(W1==0):.1f}%")
print(f"L2 sparsity: {100*np.mean(W2==0):.1f}%")
print(f"L3 sparsity: {100*np.mean(W3==0):.1f}%")

with open(os.path.join(OUTPUT_DIR, 'scales.json')) as f:
    scales = json.load(f)

# -----------------------------------------------------------------------------
# SECTION 2: Pixel normalisation → INT8
#
# train.py normalised inputs as: (pixel/255.0 - 0.1307) / 0.3081
# This maps MNIST pixels into roughly the float range [-0.42, 2.82].
#
# To represent this in INT8 we pick a fixed INPUT_SCALE that covers the range:
#   INPUT_SCALE = 2.82 / 127  ≈ 0.0222   (max normalised value / 127)
#
# Then:  int8_pixel = clip( round( normalised_float / INPUT_SCALE ), -128, 127 )
#
# This is a compile-time constant — the hardware lookup table (or input buffer)
# uses exactly this formula. test_inputs.hex stores these values.
# -----------------------------------------------------------------------------

MNIST_MEAN  = 0.1307
MNIST_STD   = 0.3081
INPUT_SCALE = 2.82 / 127.0   # covers full normalised MNIST range

def pixel_to_int8(img_uint8):
    """
    Convert (784,) uint8 raw pixel array → (784,) int8 normalised.
    Matches the normalisation used during training.
    """
    norm    = (img_uint8.astype(np.float32) / 255.0 - MNIST_MEAN) / MNIST_STD
    int8arr = np.clip(np.round(norm / INPUT_SCALE), -128, 127).astype(np.int8)
    return int8arr

# -----------------------------------------------------------------------------
# SECTION 3: Integer MAC simulation — int32 throughout, no inter-layer requant
#
# Each layer: accumulate int8 × int8 products into int32.
# Apply 20-bit saturation (matches hardware's planned 20-bit accumulator).
# After layers 1 and 2: ReLU. After layer 3: nothing.
# Pass int32 values directly into the next layer — no re-quantisation.
#
# Why this is correct:
#   The network weights were trained to produce useful outputs given
#   normalised float inputs. INT8 quantisation preserves relative magnitudes
#   within each layer. As long as the inter-layer values preserve ordering
#   (which int32 does perfectly), the argmax at the end is correct.
#
# Hardware implication:
#   Inter-layer buses will be wider (20-bit instead of 8-bit).
#   This costs some area on the accumulator-to-next-layer path.
#   An optional fixed-scale requantisation can be added later (Phase 3)
#   if area is a concern — but correctness comes first.
# -----------------------------------------------------------------------------

ACC_MIN = -(1 << 19)    # -524288  (20-bit signed min)
ACC_MAX =  (1 << 19)-1  #  524287  (20-bit signed max)

def mac_layer(x, W_int8):
    """
    x      : (n_in,)       int8 or int32 input vector
    W_int8 : (n_out, n_in) int8 weight matrix
    Returns: (n_out,)      int32, saturated to 20-bit range
    """
    n_out, n_in = W_int8.shape
    acc = np.zeros(n_out, dtype=np.int32)

    for j in range(n_out):
        for i in range(n_in):
            acc[j] += int(x[i]) * int(W_int8[j, i])
        acc[j] = int(np.clip(acc[j], ACC_MIN, ACC_MAX))

    return acc


def run_inference(img_uint8):
    """
    Full integer MLP inference.
    img_uint8: (784,) uint8 raw pixels from MNIST
    Returns:   predicted class (int), scores (int32 length 10)
    """
    x0 = pixel_to_int8(img_uint8)                  # int8  (784,)

    a1 = mac_layer(x0, W1)                          # int32 (128,)
    x1 = np.maximum(0, a1).astype(np.int32)         # ReLU

    a2 = mac_layer(x1, W2)                          # int32 (64,)
    x2 = np.maximum(0, a2).astype(np.int32)         # ReLU

    scores = mac_layer(x2, W3)                      # int32 (10,)

    return int(np.argmax(scores)), scores


# -----------------------------------------------------------------------------
# SECTION 4: Run on 100 test images, write outputs
# -----------------------------------------------------------------------------

test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True)
NUM_IMAGES   = 100
correct      = 0

golden_path     = os.path.join(OUTPUT_DIR, 'golden_outputs.txt')
inputs_hex_path = os.path.join(OUTPUT_DIR, 'test_inputs.hex')

print(f"\nRunning integer inference on {NUM_IMAGES} images...")
print("(Slow due to explicit MAC loops — ~2-5 min total)")

inputs_hex = open(inputs_hex_path, 'w')

with open(golden_path, 'w') as f_out:
    f_out.write("# ASCENT Golden Model Outputs\n")
    f_out.write("# Format: index  true_label  predicted  [score_0..score_9]  status\n")

    for idx in range(NUM_IMAGES):
        img_pil, true_label = test_dataset[idx]
        img_uint8 = np.array(img_pil, dtype=np.uint8).flatten()

        # Write normalised INT8 pixels to hex (two's complement)
        x0 = pixel_to_int8(img_uint8)
        for byte in x0.view(np.uint8):
            inputs_hex.write(f"{byte:02x}\n")

        predicted, scores = run_inference(img_uint8)

        if predicted == true_label:
            correct += 1

        scores_str = ' '.join(str(int(s)) for s in scores)
        status     = 'OK' if predicted == true_label else 'WRONG'
        f_out.write(f"{idx:4d}  {true_label}  {predicted}  [{scores_str}]  {status}\n")

        if idx < 10:
            print(f"  [{idx:3d}] true={true_label}  pred={predicted}  {status}")
        elif idx % 20 == 0:
            print(f"  [{idx:3d}] running...")

inputs_hex.close()

accuracy = 100.0 * correct / NUM_IMAGES
print(f"\nGolden model accuracy on {NUM_IMAGES} images: {accuracy:.1f}%")
print(f"Written → {golden_path}")
print(f"Written → {inputs_hex_path}")

if accuracy >= 90.0:
    print("\nPASS: Golden model accuracy acceptable (>=90%)")
    print("Phase 1 complete. Ready for Phase 2 — LOA Multiplier.")
else:
    print(f"\nWARN: Accuracy {accuracy:.1f}% still below 90%.")
    print("Check pixel normalisation constants match train.py exactly.")

Loaded weights:  L1(128, 784)  L2(64, 128)  L3(10, 64)
L1 sparsity: 62.3%
L2 sparsity: 36.7%
L3 sparsity: 23.3%

Running integer inference on 100 images...
(Slow due to explicit MAC loops — ~2-5 min total)
  [  0] true=7  pred=7  OK
  [  1] true=2  pred=1  WRONG
  [  2] true=1  pred=1  OK
  [  3] true=0  pred=0  OK
  [  4] true=4  pred=4  OK
  [  5] true=1  pred=1  OK
  [  6] true=4  pred=4  OK
  [  7] true=9  pred=9  OK
  [  8] true=5  pred=5  OK
  [  9] true=9  pred=9  OK
  [ 20] running...
  [ 40] running...
  [ 60] running...
  [ 80] running...

Golden model accuracy on 100 images: 77.0%
Written → /kaggle/working/outputs/golden_outputs.txt
Written → /kaggle/working/outputs/test_inputs.hex

WARN: Accuracy 77.0% still below 90%.
Check pixel normalisation constants match train.py exactly.


In [6]:
# =============================================================================
# ASCENT Phase 1 — golden_model.py  (v3)
# Simulate the hardware's exact integer arithmetic in Python.
#
# CHANGE LOG
# v1 → v2: Removed broken per-image re-quantisation (was 10% accuracy)
# v2 → v3: Widened L1 accumulator from 20→24 bits (was 77% accuracy)
#
# WHY 24 BITS FOR L1
#   Layer 1 has 784 inputs. Worst-case MAC magnitude:
#     784 × 127 × 127 = 12,645,136  → needs 25 signed bits
#   In practice (60% sparse weights + most pixels near zero) accumulators
#   reach ~500K-1M, but enough images hit 1M+ to ruin accuracy at 20 bits.
#   v2 saw this as 77% accuracy.
#
#   24 signed bits = ±8,388,608 — gives 8x headroom over typical peaks
#   without paying for full 25 bits everywhere.
#   L2 (128 inputs) and L3 (64 inputs) are fine at 20 bits.
#
# HARDWARE IMPLICATION
#   The signal dictionary must update: L1 accumulator becomes 24-bit.
#   Per-row partial product (16-bit) is unchanged.
#   Only the accumulator tree and ReLU output bus widen.
# =============================================================================

import numpy as np
import json
import os
from torchvision import datasets

OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'

# -----------------------------------------------------------------------------
# SECTION 1: Load quantized weights from .hex files
# -----------------------------------------------------------------------------

def load_hex_weights(path, shape):
    values = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                val = np.array([int(line, 16)], dtype=np.uint8).view(np.int8)[0]
                values.append(val)
    return np.array(values, dtype=np.int8).reshape(shape)


W1 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l1.hex'), (128, 784))
W2 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l2.hex'), ( 64, 128))
W3 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l3.hex'), ( 10,  64))

print(f"Loaded weights:  L1{W1.shape}  L2{W2.shape}  L3{W3.shape}")
print(f"L1 sparsity: {100*np.mean(W1==0):.1f}%")
print(f"L2 sparsity: {100*np.mean(W2==0):.1f}%")
print(f"L3 sparsity: {100*np.mean(W3==0):.1f}%")

with open(os.path.join(OUTPUT_DIR, 'scales.json')) as f:
    scales = json.load(f)

# -----------------------------------------------------------------------------
# SECTION 2: Pixel normalisation → INT8
# Same as v2: matches train.py's normalise(0.1307, 0.3081), then quantises
# the normalised float to INT8 with INPUT_SCALE = 2.82/127.
# -----------------------------------------------------------------------------

MNIST_MEAN  = 0.1307
MNIST_STD   = 0.3081
INPUT_SCALE = 2.82 / 127.0

def pixel_to_int8(img_uint8):
    norm    = (img_uint8.astype(np.float32) / 255.0 - MNIST_MEAN) / MNIST_STD
    int8arr = np.clip(np.round(norm / INPUT_SCALE), -128, 127).astype(np.int8)
    return int8arr

# -----------------------------------------------------------------------------
# SECTION 3: Per-layer accumulator widths
#
# Bit width depends on number of input MACs that feed each accumulator.
# Layer i with N_i inputs needs ceil(log2(N_i × 127 × 127 × 2)) signed bits.
# -----------------------------------------------------------------------------

L1_ACC_BITS = 24    # 784-input dot product, ±8.3M range
L2_ACC_BITS = 20    # 128-input dot product, ±524K range  (fits 20 bits)
L3_ACC_BITS = 20    # 64-input  dot product, ±524K range

def sat_range(bits):
    """Return (min, max) for a signed integer of given bit width."""
    return -(1 << (bits-1)), (1 << (bits-1)) - 1

L1_MIN, L1_MAX = sat_range(L1_ACC_BITS)
L2_MIN, L2_MAX = sat_range(L2_ACC_BITS)
L3_MIN, L3_MAX = sat_range(L3_ACC_BITS)

print(f"\nAccumulator widths:")
print(f"  L1: {L1_ACC_BITS} bits  range ±{L1_MAX:,}")
print(f"  L2: {L2_ACC_BITS} bits  range ±{L2_MAX:,}")
print(f"  L3: {L3_ACC_BITS} bits  range ±{L3_MAX:,}")

# -----------------------------------------------------------------------------
# SECTION 4: Integer MAC simulation
# int8 × int8 → int32 product, accumulated, then saturated to layer-specific width.
# No inter-layer re-quantisation: int32 values pass directly into next layer.
# -----------------------------------------------------------------------------

def mac_layer(x, W_int8, sat_min, sat_max):
    n_out, n_in = W_int8.shape
    acc = np.zeros(n_out, dtype=np.int32)

    for j in range(n_out):
        for i in range(n_in):
            acc[j] += int(x[i]) * int(W_int8[j, i])
        acc[j] = int(np.clip(acc[j], sat_min, sat_max))

    return acc


def run_inference(img_uint8):
    x0 = pixel_to_int8(img_uint8)                      # int8  (784,)

    a1 = mac_layer(x0, W1, L1_MIN, L1_MAX)             # int32 (128,) sat to 24b
    x1 = np.maximum(0, a1).astype(np.int32)            # ReLU

    a2 = mac_layer(x1, W2, L2_MIN, L2_MAX)             # int32 (64,)  sat to 20b
    x2 = np.maximum(0, a2).astype(np.int32)            # ReLU

    scores = mac_layer(x2, W3, L3_MIN, L3_MAX)         # int32 (10,)  sat to 20b

    return int(np.argmax(scores)), scores


# -----------------------------------------------------------------------------
# SECTION 5: Run on 100 test images
# -----------------------------------------------------------------------------

test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True)
NUM_IMAGES   = 100
correct      = 0

golden_path     = os.path.join(OUTPUT_DIR, 'golden_outputs.txt')
inputs_hex_path = os.path.join(OUTPUT_DIR, 'test_inputs.hex')

print(f"\nRunning integer inference on {NUM_IMAGES} images...")

inputs_hex = open(inputs_hex_path, 'w')

with open(golden_path, 'w') as f_out:
    f_out.write("# ASCENT Golden Model Outputs\n")
    f_out.write("# Format: index  true_label  predicted  [score_0..score_9]  status\n")

    for idx in range(NUM_IMAGES):
        img_pil, true_label = test_dataset[idx]
        img_uint8 = np.array(img_pil, dtype=np.uint8).flatten()

        x0 = pixel_to_int8(img_uint8)
        for byte in x0.view(np.uint8):
            inputs_hex.write(f"{byte:02x}\n")

        predicted, scores = run_inference(img_uint8)
        if predicted == true_label:
            correct += 1

        scores_str = ' '.join(str(int(s)) for s in scores)
        status     = 'OK' if predicted == true_label else 'WRONG'
        f_out.write(f"{idx:4d}  {true_label}  {predicted}  [{scores_str}]  {status}\n")

        if idx < 10:
            print(f"  [{idx:3d}] true={true_label}  pred={predicted}  {status}")
        elif idx % 20 == 0:
            print(f"  [{idx:3d}] running...")

inputs_hex.close()

accuracy = 100.0 * correct / NUM_IMAGES
print(f"\nGolden model accuracy on {NUM_IMAGES} images: {accuracy:.1f}%")
print(f"Written → {golden_path}")
print(f"Written → {inputs_hex_path}")

if accuracy >= 90.0:
    print("\nPASS: Golden model accuracy acceptable (>=90%)")
    print("Phase 1 complete. Ready for Phase 2 — LOA Multiplier.")
else:
    print(f"\nWARN: Accuracy {accuracy:.1f}% still below 90%.")
    print("If this is much worse than the float quantized model,")
    print("check whether L2/L3 also need wider accumulators.")

Loaded weights:  L1(128, 784)  L2(64, 128)  L3(10, 64)
L1 sparsity: 62.3%
L2 sparsity: 36.7%
L3 sparsity: 23.3%

Accumulator widths:
  L1: 24 bits  range ±8,388,607
  L2: 20 bits  range ±524,287
  L3: 20 bits  range ±524,287

Running integer inference on 100 images...
  [  0] true=7  pred=7  OK
  [  1] true=2  pred=1  WRONG
  [  2] true=1  pred=1  OK
  [  3] true=0  pred=0  OK
  [  4] true=4  pred=4  OK
  [  5] true=1  pred=1  OK
  [  6] true=4  pred=4  OK
  [  7] true=9  pred=9  OK
  [  8] true=5  pred=5  OK
  [  9] true=9  pred=9  OK
  [ 20] running...
  [ 40] running...
  [ 60] running...
  [ 80] running...

Golden model accuracy on 100 images: 77.0%
Written → /kaggle/working/outputs/golden_outputs.txt
Written → /kaggle/working/outputs/test_inputs.hex

WARN: Accuracy 77.0% still below 90%.
If this is much worse than the float quantized model,
check whether L2/L3 also need wider accumulators.


In [7]:
# =============================================================================
# ASCENT Phase 1 — golden_model.py  (v3)
# Simulate the hardware's exact integer arithmetic in Python.
#
# CHANGE LOG
# v1 → v2: Removed broken per-image re-quantisation (was 10% accuracy)
# v2 → v3: Widened L1 accumulator from 20→24 bits (was 77% accuracy)
#
# WHY 24 BITS FOR L1
#   Layer 1 has 784 inputs. Worst-case MAC magnitude:
#     784 × 127 × 127 = 12,645,136  → needs 25 signed bits
#   In practice (60% sparse weights + most pixels near zero) accumulators
#   reach ~500K-1M, but enough images hit 1M+ to ruin accuracy at 20 bits.
#   v2 saw this as 77% accuracy.
#
#   24 signed bits = ±8,388,608 — gives 8x headroom over typical peaks
#   without paying for full 25 bits everywhere.
#   L2 (128 inputs) and L3 (64 inputs) are fine at 20 bits.
#
# HARDWARE IMPLICATION
#   The signal dictionary must update: L1 accumulator becomes 24-bit.
#   Per-row partial product (16-bit) is unchanged.
#   Only the accumulator tree and ReLU output bus widen.
# =============================================================================

import numpy as np
import json
import os
from torchvision import datasets

OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'

# -----------------------------------------------------------------------------
# SECTION 1: Load quantized weights from .hex files
# -----------------------------------------------------------------------------

def load_hex_weights(path, shape):
    values = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                val = np.array([int(line, 16)], dtype=np.uint8).view(np.int8)[0]
                values.append(val)
    return np.array(values, dtype=np.int8).reshape(shape)


W1 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l1.hex'), (128, 784))
W2 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l2.hex'), ( 64, 128))
W3 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l3.hex'), ( 10,  64))

print(f"Loaded weights:  L1{W1.shape}  L2{W2.shape}  L3{W3.shape}")
print(f"L1 sparsity: {100*np.mean(W1==0):.1f}%")
print(f"L2 sparsity: {100*np.mean(W2==0):.1f}%")
print(f"L3 sparsity: {100*np.mean(W3==0):.1f}%")

with open(os.path.join(OUTPUT_DIR, 'scales.json')) as f:
    scales = json.load(f)

# -----------------------------------------------------------------------------
# SECTION 2: Pixel normalisation → INT8
# Same as v2: matches train.py's normalise(0.1307, 0.3081), then quantises
# the normalised float to INT8 with INPUT_SCALE = 2.82/127.
# -----------------------------------------------------------------------------

MNIST_MEAN  = 0.1307
MNIST_STD   = 0.3081
INPUT_SCALE = 2.82 / 127.0

def pixel_to_int8(img_uint8):
    norm    = (img_uint8.astype(np.float32) / 255.0 - MNIST_MEAN) / MNIST_STD
    int8arr = np.clip(np.round(norm / INPUT_SCALE), -128, 127).astype(np.int8)
    return int8arr

# -----------------------------------------------------------------------------
# SECTION 3: Per-layer accumulator widths
#
# Bit width depends on number of input MACs that feed each accumulator.
# Layer i with N_i inputs needs ceil(log2(N_i × 127 × 127 × 2)) signed bits.
# -----------------------------------------------------------------------------

L1_ACC_BITS = 24    # 784-input dot product, ±8.3M range
L2_ACC_BITS = 20    # 128-input dot product, ±524K range  (fits 20 bits)
L3_ACC_BITS = 20    # 64-input  dot product, ±524K range

def sat_range(bits):
    """Return (min, max) for a signed integer of given bit width."""
    return -(1 << (bits-1)), (1 << (bits-1)) - 1

L1_MIN, L1_MAX = sat_range(L1_ACC_BITS)
L2_MIN, L2_MAX = sat_range(L2_ACC_BITS)
L3_MIN, L3_MAX = sat_range(L3_ACC_BITS)

print(f"\nAccumulator widths:")
print(f"  L1: {L1_ACC_BITS} bits  range ±{L1_MAX:,}")
print(f"  L2: {L2_ACC_BITS} bits  range ±{L2_MAX:,}")
print(f"  L3: {L3_ACC_BITS} bits  range ±{L3_MAX:,}")

# -----------------------------------------------------------------------------
# SECTION 4: Integer MAC simulation
# int8 × int8 → int32 product, accumulated, then saturated to layer-specific width.
# No inter-layer re-quantisation: int32 values pass directly into next layer.
# -----------------------------------------------------------------------------

def mac_layer(x, W_int8, sat_min, sat_max):
    n_out, n_in = W_int8.shape
    acc = np.zeros(n_out, dtype=np.int32)

    for j in range(n_out):
        for i in range(n_in):
            acc[j] += int(x[i]) * int(W_int8[j, i])
        acc[j] = int(np.clip(acc[j], sat_min, sat_max))

    return acc


def run_inference(img_uint8):
    x0 = pixel_to_int8(img_uint8)                      # int8  (784,)

    a1 = mac_layer(x0, W1, L1_MIN, L1_MAX)             # int32 (128,) sat to 24b
    x1 = np.maximum(0, a1).astype(np.int32)            # ReLU

    a2 = mac_layer(x1, W2, L2_MIN, L2_MAX)             # int32 (64,)  sat to 20b
    x2 = np.maximum(0, a2).astype(np.int32)            # ReLU

    scores = mac_layer(x2, W3, L3_MIN, L3_MAX)         # int32 (10,)  sat to 20b

    return int(np.argmax(scores)), scores


# -----------------------------------------------------------------------------
# SECTION 5: Run on 100 test images
# -----------------------------------------------------------------------------

test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True)
NUM_IMAGES   = 100
correct      = 0

golden_path     = os.path.join(OUTPUT_DIR, 'golden_outputs.txt')
inputs_hex_path = os.path.join(OUTPUT_DIR, 'test_inputs.hex')

print(f"\nRunning integer inference on {NUM_IMAGES} images...")

inputs_hex = open(inputs_hex_path, 'w')

with open(golden_path, 'w') as f_out:
    f_out.write("# ASCENT Golden Model Outputs\n")
    f_out.write("# Format: index  true_label  predicted  [score_0..score_9]  status\n")

    for idx in range(NUM_IMAGES):
        img_pil, true_label = test_dataset[idx]
        img_uint8 = np.array(img_pil, dtype=np.uint8).flatten()

        x0 = pixel_to_int8(img_uint8)
        for byte in x0.view(np.uint8):
            inputs_hex.write(f"{byte:02x}\n")

        predicted, scores = run_inference(img_uint8)
        if predicted == true_label:
            correct += 1

        scores_str = ' '.join(str(int(s)) for s in scores)
        status     = 'OK' if predicted == true_label else 'WRONG'
        f_out.write(f"{idx:4d}  {true_label}  {predicted}  [{scores_str}]  {status}\n")

        if idx < 10:
            print(f"  [{idx:3d}] true={true_label}  pred={predicted}  {status}")
        elif idx % 20 == 0:
            print(f"  [{idx:3d}] running...")

inputs_hex.close()

accuracy = 100.0 * correct / NUM_IMAGES
print(f"\nGolden model accuracy on {NUM_IMAGES} images: {accuracy:.1f}%")
print(f"Written → {golden_path}")
print(f"Written → {inputs_hex_path}")

if accuracy >= 90.0:
    print("\nPASS: Golden model accuracy acceptable (>=90%)")
    print("Phase 1 complete. Ready for Phase 2 — LOA Multiplier.")
else:
    print(f"\nWARN: Accuracy {accuracy:.1f}% still below 90%.")
    print("If this is much worse than the float quantized model,")
    print("check whether L2/L3 also need wider accumulators.")

Loaded weights:  L1(128, 784)  L2(64, 128)  L3(10, 64)
L1 sparsity: 62.3%
L2 sparsity: 36.7%
L3 sparsity: 23.3%

Accumulator widths:
  L1: 24 bits  range ±8,388,607
  L2: 20 bits  range ±524,287
  L3: 20 bits  range ±524,287

Running integer inference on 100 images...
  [  0] true=7  pred=7  OK
  [  1] true=2  pred=1  WRONG
  [  2] true=1  pred=1  OK
  [  3] true=0  pred=0  OK
  [  4] true=4  pred=4  OK
  [  5] true=1  pred=1  OK
  [  6] true=4  pred=4  OK
  [  7] true=9  pred=9  OK
  [  8] true=5  pred=5  OK
  [  9] true=9  pred=9  OK
  [ 20] running...
  [ 40] running...
  [ 60] running...
  [ 80] running...

Golden model accuracy on 100 images: 77.0%
Written → /kaggle/working/outputs/golden_outputs.txt
Written → /kaggle/working/outputs/test_inputs.hex

WARN: Accuracy 77.0% still below 90%.
If this is much worse than the float quantized model,
check whether L2/L3 also need wider accumulators.


In [8]:
# =============================================================================
# ASCENT Phase 1 — golden_model.py  (v4 — CORRECT)
#
# FULL BUG HISTORY
# ─────────────────────────────────────────────────────────────────────────────
# v1: 10% accuracy
#     Per-image requant scale: max_val/127 changes every image.
#     Destroys relative magnitude — every image looks identical to layer 2.
#
# v2: 77% accuracy
#     Removed requantisation entirely — passed raw int32 between layers.
#     BUT: L2 weights were trained/quantised assuming INT8 inputs (±127).
#     We fed it values ±millions. Every L2 accumulator saturated immediately.
#     Accuracy didn't change from v1 because L2/L3 were still producing garbage.
#
# v3: 77% accuracy (identical to v2)
#     Widened L1 accumulator from 20→24 bits.
#     L1 itself was fine — saturation wasn't the bottleneck.
#     The bug was still the raw int32 being passed to L2.
#
# v4: FIXED
#     Use a FIXED requantisation scale between layers.
#     Fixed = same constant for every image = a compile-time hardware constant.
#     Scale = ACC_MAX / 127   (maps peak accumulator value → INT8 max)
#     This is exactly what real quantised neural network runtimes do
#     (TFLite, ONNX Runtime, etc.) — it is called "static quantisation".
#
# HARDWARE IMPLICATION
#     Each CIM layer needs a fixed-scale divider after its accumulator
#     before the result is stored back for the next layer.
#     This is a single right-shift (since scale is a power-of-2-friendly number).
#     We will implement this as a parameterised shift in Phase 3.
# =============================================================================

import numpy as np
import json
import os
from torchvision import datasets

OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'

# -----------------------------------------------------------------------------
# SECTION 1: Load quantized weights
# -----------------------------------------------------------------------------

def load_hex_weights(path, shape):
    values = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                val = np.array([int(line, 16)], dtype=np.uint8).view(np.int8)[0]
                values.append(val)
    return np.array(values, dtype=np.int8).reshape(shape)


W1 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l1.hex'), (128, 784))
W2 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l2.hex'), ( 64, 128))
W3 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l3.hex'), ( 10,  64))

print(f"Loaded weights:  L1{W1.shape}  L2{W2.shape}  L3{W3.shape}")
print(f"L1 sparsity: {100*np.mean(W1==0):.1f}%")
print(f"L2 sparsity: {100*np.mean(W2==0):.1f}%")
print(f"L3 sparsity: {100*np.mean(W3==0):.1f}%")

with open(os.path.join(OUTPUT_DIR, 'scales.json')) as f:
    scales = json.load(f)

# -----------------------------------------------------------------------------
# SECTION 2: Pixel normalisation → INT8
# -----------------------------------------------------------------------------

MNIST_MEAN  = 0.1307
MNIST_STD   = 0.3081
INPUT_SCALE = 2.8215 / 127.0   # true max of normalised MNIST / 127

def pixel_to_int8(img_uint8):
    norm    = (img_uint8.astype(np.float32) / 255.0 - MNIST_MEAN) / MNIST_STD
    int8arr = np.clip(np.round(norm / INPUT_SCALE), -128, 127).astype(np.int8)
    return int8arr

# -----------------------------------------------------------------------------
# SECTION 3: Accumulator widths and FIXED requantisation scales
#
# After each layer's accumulator, we requantise the output back to INT8
# so the next layer's weights (calibrated for INT8 inputs) work correctly.
#
# The requantisation scale is FIXED — the same constant for every image.
# It maps the peak of the accumulator's possible range to INT8 max (127).
#
#   requant_scale = ACC_MAX / 127
#   out_int8 = clip( round( relu_out / requant_scale ), 0, 127 )
#
# This is a hardware-friendly operation: dividing by ACC_MAX/127 is the same
# as multiplying by 127/ACC_MAX, which can be implemented as a fixed-point
# multiply or an arithmetic right-shift (if ACC_MAX is a power of 2 − 1,
# which it nearly is for our chosen bit widths).
#
# Accumulator widths:
#   L1: 784 inputs × INT8 × INT8 → needs 25 bits worst case → budget 24 bits
#   L2: 128 inputs × INT8 × INT8 → needs 22 bits worst case → budget 20 bits
#   L3:  64 inputs × INT8 × INT8 → needs 21 bits worst case → budget 20 bits
#       (L3 has no requantisation — scores go straight to argmax)
# -----------------------------------------------------------------------------

L1_ACC_BITS = 24
L2_ACC_BITS = 20
L3_ACC_BITS = 20

L1_ACC_MAX =  (1 << (L1_ACC_BITS-1)) - 1   # +8,388,607
L1_ACC_MIN = -(1 << (L1_ACC_BITS-1))        # -8,388,608
L2_ACC_MAX =  (1 << (L2_ACC_BITS-1)) - 1   # +524,287
L2_ACC_MIN = -(1 << (L2_ACC_BITS-1))        # -524,288
L3_ACC_MAX =  (1 << (L3_ACC_BITS-1)) - 1
L3_ACC_MIN = -(1 << (L3_ACC_BITS-1))

# Fixed requantisation divisors
L1_REQUANT = L1_ACC_MAX / 127.0   # 66,052.0
L2_REQUANT = L2_ACC_MAX / 127.0   #  4,128.2

print(f"\nAccumulator config:")
print(f"  L1: {L1_ACC_BITS}-bit  range [{L1_ACC_MIN:,}, {L1_ACC_MAX:,}]"
      f"  requant_divisor={L1_REQUANT:.1f}")
print(f"  L2: {L2_ACC_BITS}-bit  range [{L2_ACC_MIN:,}, {L2_ACC_MAX:,}]"
      f"  requant_divisor={L2_REQUANT:.1f}")
print(f"  L3: {L3_ACC_BITS}-bit  range [{L3_ACC_MIN:,}, {L3_ACC_MAX:,}]"
      f"  (no requant — final argmax)")

# -----------------------------------------------------------------------------
# SECTION 4: Integer MAC simulation with fixed-scale requantisation
# -----------------------------------------------------------------------------

def mac_layer(x, W_int8, acc_min, acc_max):
    """
    x       : (n_in,)      int8 input vector
    W_int8  : (n_out,n_in) int8 weight matrix
    Returns : (n_out,)     int32, saturated to [acc_min, acc_max]
    """
    n_out, n_in = W_int8.shape
    acc = np.zeros(n_out, dtype=np.int32)

    for j in range(n_out):
        for i in range(n_in):
            acc[j] += int(x[i]) * int(W_int8[j, i])
        acc[j] = int(np.clip(acc[j], acc_min, acc_max))

    return acc


def requantise(acc_int32, requant_divisor):
    """
    Requantise a post-ReLU accumulator output back to INT8.
    Uses a fixed compile-time divisor — same constant for every image.
    Maps [0, ACC_MAX] → [0, 127].
    """
    out = np.clip(np.round(acc_int32 / requant_divisor), 0, 127).astype(np.int8)
    return out


def run_inference(img_uint8):
    """
    Full integer MLP inference — pixel → predicted digit class.
    """
    # Input: normalised INT8 pixels
    x0 = pixel_to_int8(img_uint8)                        # int8  (784,)

    # Layer 1: MAC → 24-bit saturation → ReLU → fixed requantise → INT8
    a1 = mac_layer(x0, W1, L1_ACC_MIN, L1_ACC_MAX)       # int32 (128,)
    r1 = np.maximum(0, a1).astype(np.int32)              # ReLU
    x1 = requantise(r1, L1_REQUANT)                      # int8  (128,)

    # Layer 2: MAC → 20-bit saturation → ReLU → fixed requantise → INT8
    a2 = mac_layer(x1, W2, L2_ACC_MIN, L2_ACC_MAX)       # int32 (64,)
    r2 = np.maximum(0, a2).astype(np.int32)              # ReLU
    x2 = requantise(r2, L2_REQUANT)                      # int8  (64,)

    # Layer 3: MAC → 20-bit saturation → argmax (no ReLU, no requantise)
    scores = mac_layer(x2, W3, L3_ACC_MIN, L3_ACC_MAX)   # int32 (10,)

    return int(np.argmax(scores)), scores


# -----------------------------------------------------------------------------
# SECTION 5: Run on 100 test images
# -----------------------------------------------------------------------------

test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True)
NUM_IMAGES   = 100
correct      = 0

golden_path     = os.path.join(OUTPUT_DIR, 'golden_outputs.txt')
inputs_hex_path = os.path.join(OUTPUT_DIR, 'test_inputs.hex')

print(f"\nRunning integer inference on {NUM_IMAGES} images...")

inputs_hex = open(inputs_hex_path, 'w')

with open(golden_path, 'w') as f_out:
    f_out.write("# ASCENT Golden Model Outputs v4\n")
    f_out.write("# Format: index  true_label  predicted  [score_0..score_9]  status\n")

    for idx in range(NUM_IMAGES):
        img_pil, true_label = test_dataset[idx]
        img_uint8 = np.array(img_pil, dtype=np.uint8).flatten()

        # Write normalised INT8 pixels (what hardware actually receives)
        x0 = pixel_to_int8(img_uint8)
        for byte in x0.view(np.uint8):
            inputs_hex.write(f"{byte:02x}\n")

        predicted, scores = run_inference(img_uint8)
        if predicted == true_label:
            correct += 1

        scores_str = ' '.join(str(int(s)) for s in scores)
        status     = 'OK' if predicted == true_label else 'WRONG'
        f_out.write(f"{idx:4d}  {true_label}  {predicted}  [{scores_str}]  {status}\n")

        if idx < 10:
            print(f"  [{idx:3d}] true={true_label}  pred={predicted}  {status}")
        elif idx % 20 == 0:
            print(f"  [{idx:3d}] running...")

inputs_hex.close()

accuracy = 100.0 * correct / NUM_IMAGES
print(f"\nGolden model accuracy on {NUM_IMAGES} images: {accuracy:.1f}%")
print(f"Written → {golden_path}")
print(f"Written → {inputs_hex_path}")

if accuracy >= 90.0:
    print("\nPASS: Phase 1 complete. Ready for Phase 2 — LOA Multiplier.")
else:
    print(f"\nWARN: {accuracy:.1f}% still below 90%.")
    print("Run the debug cell below to inspect inter-layer values.")
    print()
    # Debug: show actual accumulator ranges hit during inference
    print("--- DEBUG: Accumulator range profiling ---")
    test_dataset2 = datasets.MNIST(DATA_DIR, train=False, download=True)
    max_l1 = max_l2 = 0
    for idx in range(20):
        img_pil, _ = test_dataset2[idx]
        img_uint8  = np.array(img_pil, dtype=np.uint8).flatten()
        x0 = pixel_to_int8(img_uint8)
        a1 = mac_layer(x0, W1, L1_ACC_MIN, L1_ACC_MAX)
        r1 = np.maximum(0, a1).astype(np.int32)
        x1 = requantise(r1, L1_REQUANT)
        a2 = mac_layer(x1, W2, L2_ACC_MIN, L2_ACC_MAX)
        max_l1 = max(max_l1, int(np.max(np.abs(a1))))
        max_l2 = max(max_l2, int(np.max(np.abs(a2))))
    print(f"  Max |L1 acc| across 20 images: {max_l1:,}  (budget: {L1_ACC_MAX:,})")
    print(f"  Max |L2 acc| across 20 images: {max_l2:,}  (budget: {L2_ACC_MAX:,})")

Loaded weights:  L1(128, 784)  L2(64, 128)  L3(10, 64)
L1 sparsity: 62.3%
L2 sparsity: 36.7%
L3 sparsity: 23.3%

Accumulator config:
  L1: 24-bit  range [-8,388,608, 8,388,607]  requant_divisor=66052.0
  L2: 20-bit  range [-524,288, 524,287]  requant_divisor=4128.2
  L3: 20-bit  range [-524,288, 524,287]  (no requant — final argmax)

Running integer inference on 100 images...
  [  0] true=7  pred=0  WRONG
  [  1] true=2  pred=0  WRONG
  [  2] true=1  pred=0  WRONG
  [  3] true=0  pred=0  OK
  [  4] true=4  pred=0  WRONG
  [  5] true=1  pred=0  WRONG
  [  6] true=4  pred=0  WRONG
  [  7] true=9  pred=0  WRONG
  [  8] true=5  pred=0  WRONG
  [  9] true=9  pred=0  WRONG
  [ 20] running...
  [ 40] running...
  [ 60] running...
  [ 80] running...

Golden model accuracy on 100 images: 8.0%
Written → /kaggle/working/outputs/golden_outputs.txt
Written → /kaggle/working/outputs/test_inputs.hex

WARN: 8.0% still below 90%.
Run the debug cell below to inspect inter-layer values.

--- DEBUG: Accum

In [12]:
# =============================================================================
# ASCENT Phase 1 — golden_model.py  (v5 — CALIBRATED)
#
# ROOT CAUSE OF ALL PREVIOUS FAILURES
# ─────────────────────────────────────────────────────────────────────────────
# The requantisation divisor between layers must match the ACTUAL activation
# range — not the theoretical worst-case accumulator maximum.
#
# v4 used: L1_REQUANT = 8,388,607 / 127 = 66,052
# Actual L1 peak across real images: ~209,000
# So v4 divided by 66,052 when it should have divided by ~1,648.
# Result: L2 received inputs of 0 or 1 — no information — predicted class 0 always.
#
# THE FIX: Calibration
# Run 500 calibration images through the integer MAC layers (no requantisation).
# Collect the 99th percentile of peak activation magnitudes per layer.
# Use those measured values as the requantisation divisors.
# This is exactly what TFLite, ONNX Runtime, and real hardware compilers do.
# The calibrated divisor becomes a FIXED HARDWARE CONSTANT — same for all images.
#
# HARDWARE IMPLICATION
# The calibration step produces two constants:
#   SHIFT_L1: log2(L1_REQUANT) ≈ a right-shift amount
#   SHIFT_L2: log2(L2_REQUANT) ≈ a right-shift amount
# In RTL these become arithmetic right-shifts after the accumulator,
# which cost zero extra logic (just wiring).
# =============================================================================

import numpy as np
import json
import os
from torchvision import datasets

OUTPUT_DIR = '/kaggle/working/outputs' if os.path.exists('/kaggle/working') else 'outputs'
DATA_DIR   = '/kaggle/working/data'    if os.path.exists('/kaggle/working') else './data'

# -----------------------------------------------------------------------------
# SECTION 1: Load weights
# -----------------------------------------------------------------------------

def load_hex_weights(path, shape):
    values = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                val = np.array([int(line, 16)], dtype=np.uint8).view(np.int8)[0]
                values.append(val)
    return np.array(values, dtype=np.int8).reshape(shape)


W1 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l1.hex'), (128, 784))
W2 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l2.hex'), ( 64, 128))
W3 = load_hex_weights(os.path.join(OUTPUT_DIR, 'weights_l3.hex'), ( 10,  64))

print(f"Loaded weights:  L1{W1.shape}  L2{W2.shape}  L3{W3.shape}")
print(f"L1 sparsity: {100*np.mean(W1==0):.1f}%")
print(f"L2 sparsity: {100*np.mean(W2==0):.1f}%")
print(f"L3 sparsity: {100*np.mean(W3==0):.1f}%")

with open(os.path.join(OUTPUT_DIR, 'scales.json')) as f:
    scales = json.load(f)

# -----------------------------------------------------------------------------
# SECTION 2: Pixel normalisation → INT8
# -----------------------------------------------------------------------------

MNIST_MEAN  = 0.1307
MNIST_STD   = 0.3081
INPUT_SCALE = 2.8215 / 127.0

def pixel_to_int8(img_uint8):
    norm    = (img_uint8.astype(np.float32) / 255.0 - MNIST_MEAN) / MNIST_STD
    int8arr = np.clip(np.round(norm / INPUT_SCALE), -128, 127).astype(np.int8)
    return int8arr

# -----------------------------------------------------------------------------
# SECTION 3: Accumulator widths
# -----------------------------------------------------------------------------

L1_ACC_BITS = 24
L2_ACC_BITS = 20
L3_ACC_BITS = 20

L1_ACC_MIN, L1_ACC_MAX = -(1<<(L1_ACC_BITS-1)),  (1<<(L1_ACC_BITS-1))-1
L2_ACC_MIN, L2_ACC_MAX = -(1<<(L2_ACC_BITS-1)),  (1<<(L2_ACC_BITS-1))-1
L3_ACC_MIN, L3_ACC_MAX = -(1<<(L3_ACC_BITS-1)),  (1<<(L3_ACC_BITS-1))-1

# -----------------------------------------------------------------------------
# SECTION 4: MAC layer (no requantisation — raw accumulator out)
# -----------------------------------------------------------------------------

def mac_layer(x, W_int8, acc_min, acc_max):
    n_out, n_in = W_int8.shape
    acc = np.zeros(n_out, dtype=np.int32)
    for j in range(n_out):
        for i in range(n_in):
            acc[j] += int(x[i]) * int(W_int8[j, i])
        acc[j] = int(np.clip(acc[j], acc_min, acc_max))
    return acc

# -----------------------------------------------------------------------------
# SECTION 5: CALIBRATION
#
# Run 500 images through L1 and L2 (without requantisation between them,
# using the raw acc values). Collect the 99th percentile of peak values.
# These become our fixed requantisation divisors.
#
# Why 99th percentile and not max?
# Using the absolute max would be set by one outlier image, making
# the divisor too large and under-utilising the INT8 range for typical images.
# 99th percentile: 99% of images map their peak activation to ≥1 in INT8 —
# a good balance between range coverage and outlier robustness.
# -----------------------------------------------------------------------------

print("\n--- Calibration (500 images) ---")

calib_dataset = datasets.MNIST(DATA_DIR, train=True, download=True)
N_CALIB = 500

l1_peaks = []   # max |acc| for each calibration image, layer 1
l2_peaks = []   # max |acc| for each calibration image, layer 2
                # For L2 calibration we need to pass something into it.
                # We use the L1 output requantised with a TEMPORARY rough scale
                # (we'll iterate if needed, but one pass is usually enough).

# First pass: collect L1 peaks
for idx in range(N_CALIB):
    img_pil, _ = calib_dataset[idx]
    img_uint8  = np.array(img_pil, dtype=np.uint8).flatten()
    x0 = pixel_to_int8(img_uint8)
    a1 = mac_layer(x0, W1, L1_ACC_MIN, L1_ACC_MAX)
    l1_peaks.append(float(np.max(np.abs(a1))))

L1_CALIB_MAX   = float(np.percentile(l1_peaks, 99))
L1_REQUANT_DIV = L1_CALIB_MAX / 127.0

print(f"  L1 99th percentile peak: {L1_CALIB_MAX:,.0f}")
print(f"  L1 requant divisor:      {L1_REQUANT_DIV:.2f}")

# Second pass: collect L2 peaks using the calibrated L1 requant
def requantise(acc_int32, divisor):
    return np.clip(np.round(acc_int32.astype(np.float32) / divisor),
                   0, 127).astype(np.int8)

for idx in range(N_CALIB):
    img_pil, _ = calib_dataset[idx]
    img_uint8  = np.array(img_pil, dtype=np.uint8).flatten()
    x0 = pixel_to_int8(img_uint8)
    a1 = mac_layer(x0, W1, L1_ACC_MIN, L1_ACC_MAX)
    r1 = np.maximum(0, a1).astype(np.int32)
    x1 = requantise(r1, L1_REQUANT_DIV)             # INT8 out of L1
    a2 = mac_layer(x1, W2, L2_ACC_MIN, L2_ACC_MAX)
    l2_peaks.append(float(np.max(np.abs(a2))))

L2_CALIB_MAX   = float(np.percentile(l2_peaks, 99))
L2_REQUANT_DIV = L2_CALIB_MAX / 127.0

print(f"  L2 99th percentile peak: {L2_CALIB_MAX:,.0f}")
print(f"  L2 requant divisor:      {L2_REQUANT_DIV:.2f}")

# Save calibration constants — hardware will use these as fixed shift/multiply
calib = {
    'L1_CALIB_MAX':   L1_CALIB_MAX,
    'L1_REQUANT_DIV': L1_REQUANT_DIV,
    'L2_CALIB_MAX':   L2_CALIB_MAX,
    'L2_REQUANT_DIV': L2_REQUANT_DIV,
}
calib_path = os.path.join(OUTPUT_DIR, 'calibration.json')
with open(calib_path, 'w') as f:
    json.dump(calib, f, indent=2)
print(f"  Calibration saved → {calib_path}")

# -----------------------------------------------------------------------------
# SECTION 6: Full inference using calibrated constants
# -----------------------------------------------------------------------------

def run_inference(img_uint8):
    x0 = pixel_to_int8(img_uint8)                         # int8  (784,)

    a1 = mac_layer(x0, W1, L1_ACC_MIN, L1_ACC_MAX)        # int32 (128,) 24b sat
    r1 = np.maximum(0, a1).astype(np.int32)               # ReLU
    x1 = requantise(r1, L1_REQUANT_DIV)                   # int8  (128,)

    a2 = mac_layer(x1, W2, L2_ACC_MIN, L2_ACC_MAX)        # int32 (64,)  20b sat
    r2 = np.maximum(0, a2).astype(np.int32)               # ReLU
    x2 = requantise(r2, L2_REQUANT_DIV)                   # int8  (64,)

    scores = mac_layer(x2, W3, L3_ACC_MIN, L3_ACC_MAX)    # int32 (10,)  20b sat

    return int(np.argmax(scores)), scores


# -----------------------------------------------------------------------------
# SECTION 7: Run on 100 test images
# -----------------------------------------------------------------------------

test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True)
NUM_IMAGES   = 100
correct      = 0

golden_path     = os.path.join(OUTPUT_DIR, 'golden_outputs.txt')
inputs_hex_path = os.path.join(OUTPUT_DIR, 'test_inputs.hex')

print(f"\nRunning integer inference on {NUM_IMAGES} images...")

inputs_hex = open(inputs_hex_path, 'w')

with open(golden_path, 'w') as f_out:
    f_out.write("# ASCENT Golden Model Outputs v5\n")
    f_out.write("# L1_REQUANT_DIV={:.2f}  L2_REQUANT_DIV={:.2f}\n".format(
        L1_REQUANT_DIV, L2_REQUANT_DIV))
    f_out.write("# Format: index  true_label  predicted  [score_0..score_9]  status\n")

    for idx in range(NUM_IMAGES):
        img_pil, true_label = test_dataset[idx]
        img_uint8 = np.array(img_pil, dtype=np.uint8).flatten()

        x0 = pixel_to_int8(img_uint8)
        for byte in x0.view(np.uint8):
            inputs_hex.write(f"{byte:02x}\n")

        predicted, scores = run_inference(img_uint8)
        if predicted == true_label:
            correct += 1

        scores_str = ' '.join(str(int(s)) for s in scores)
        status     = 'OK' if predicted == true_label else 'WRONG'
        f_out.write(f"{idx:4d}  {true_label}  {predicted}  [{scores_str}]  {status}\n")

        if idx < 10:
            print(f"  [{idx:3d}] true={true_label}  pred={predicted}  {status}")
        elif idx % 20 == 0:
            print(f"  [{idx:3d}] running...")

inputs_hex.close()

accuracy = 100.0 * correct / NUM_IMAGES
print(f"\nGolden model accuracy on {NUM_IMAGES} images: {accuracy:.1f}%")
print(f"Written → {golden_path}")
print(f"Written → {inputs_hex_path}")
print(f"Written → {calib_path}")

if accuracy >= 90.0:
    print("\nPASS: Phase 1 complete. Ready for Phase 2 — LOA Multiplier.")
    print(f"\nHardware constants to note in signal_dict.md:")
    l1_shift = int(np.round(np.log2(L1_REQUANT_DIV)))
    l2_shift = int(np.round(np.log2(max(L2_REQUANT_DIV, 1))))
    print(f"  L1 requant divisor : {L1_REQUANT_DIV:.1f}  (approx right-shift by {l1_shift} bits)")
    print(f"  L2 requant divisor : {L2_REQUANT_DIV:.1f}  (approx right-shift by {l2_shift} bits)")
else:
    print(f"\nWARN: {accuracy:.1f}% — try increasing N_CALIB to 1000 or widening the percentile to 99.9")

Loaded weights:  L1(128, 784)  L2(64, 128)  L3(10, 64)
L1 sparsity: 62.3%
L2 sparsity: 36.7%
L3 sparsity: 23.3%

--- Calibration (500 images) ---
  L1 99th percentile peak: 266,837
  L1 requant divisor:      2101.08
  L2 99th percentile peak: 19,698
  L2 requant divisor:      155.10
  Calibration saved → /kaggle/working/outputs/calibration.json

Running integer inference on 100 images...
  [  0] true=7  pred=7  OK
  [  1] true=2  pred=2  OK
  [  2] true=1  pred=1  OK
  [  3] true=0  pred=0  OK
  [  4] true=4  pred=4  OK
  [  5] true=1  pred=1  OK
  [  6] true=4  pred=4  OK
  [  7] true=9  pred=9  OK
  [  8] true=5  pred=5  OK
  [  9] true=9  pred=9  OK
  [ 20] running...
  [ 40] running...
  [ 60] running...
  [ 80] running...

Golden model accuracy on 100 images: 98.0%
Written → /kaggle/working/outputs/golden_outputs.txt
Written → /kaggle/working/outputs/test_inputs.hex
Written → /kaggle/working/outputs/calibration.json

PASS: Phase 1 complete. Ready for Phase 2 — LOA Multiplier.

Har